# 🐼 Тренажер по Pandas: Уровень 2 (Агрегации, Джойны, Временные ряды и ML-фичи)

Как работать с этим ноутбуком:
1. Внимательно прочитай блок **«Методы для задачи»** и **«Связанные методы в копилку»** перед заданием.
2. Замени `pass` в кодовой ячейке на свое решение.
3. Запусти ячейку (`Shift + Enter`), чтобы проверить себя на `assert`-тестах.
4. Если что-то не получается — открой спойлер с подсказкой и готовым решением.

In [1]:
# Запусти эту ячейку один раз перед началом работы!
import pandas as pd
import numpy as np
print("✅ Библиотека Pandas успешно импортирована! Версия:", pd.__version__)

✅ Библиотека Pandas успешно импортирована! Версия: 3.0.6


---
## Модуль 1. Группировка и именованная агрегация (`groupby` + Named `agg`)

### 🎯 Методы, которые нужны для задачи:
* `df.groupby('col', as_index=False)` — группирует строки по уникальным значениям колонки `'col'`.
  * Если указать `as_index=False` (или вызвать `.reset_index()` в конце), колонка группировки останется обычным столбцом, а не превратится в индекс.
* **Именованная агрегация (Named Aggregation)** в `.agg()` — самый чистый способ посчитать сразу несколько статистик по разным колонкам и сразу дать им понятные имена (без двухуровневого `MultiIndex`!):
  ```python
  df.groupby('category', as_index=False).agg(
      total_revenue=('price', 'sum'),
      avg_check=('price', 'mean'),
      unique_users=('user_id', 'nunique')
  )
  ```
* **Важное отличие `'count'` от `'size'`:**
  * `'count'` считает только **непустые (не-NaN)** значения в колонке.
  * `'size'` считает **общее число строк** в группе (включая `NaN`).

### 📚 Связанные методы в копилку:
* Внутрь кортежа `('col', func)` в `.agg()` можно передавать не только строки (`'sum'`, `'mean'`, `'median'`, `'std'`, `'min'`, `'max'`, `'first'`, `'last'`), но и любую свою функцию или `lambda`: например, `range_val=('price', lambda x: x.max() - x.min())`.
* По умолчанию `groupby` игнорирует группы, где ключ равен `NaN`. Чтобы включить `NaN` в отдельную группу, передай `dropna=False` внутрь `df.groupby('col', dropna=False)`.

### Задача 1
Напиши функцию `summarize_sales(df)`, которая принимает таблицу транзакций с колонками `'store_id'`, `'amount'` и `'customer_id'` и возвращает агрегированный `DataFrame` (с обычным числовым индексом, `store_id` должен быть колонкой!) со следующими столбцами для каждого магазина `'store_id'`:
1. `'total_sales'` — сумма покупок (`'amount'`).
2. `'max_bill'` — максимальный чек (`'amount'`).
3. `'unique_customers'` — число уникальных покупателей (`'customer_id'`).

In [2]:
sales_df = pd.DataFrame({
    "store_id": ["S1", "S1", "S2", "S1", "S2"],
    "amount": [100.0, 250.0, 500.0, 150.0, 300.0],
    "customer_id": [1, 2, 3, 1, 3]
})

    # df['total_sales'] = df.groupby('store_id')['amount'].transform('sum')
    # df['max_bill'] = df.groupby('store_id')['amount'].transform('max')
    # df['unique_customers'] = df.groupby('store_id')['amount'].transform('nunique')
    # return df

In [3]:
sales_df.groupby('store_id',as_index=False)['amount'].mean()

,store_id,amount
0,S1,166.666667
1,S2,400.000000


In [4]:
def summarize_sales(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    return df.groupby('store_id',as_index=False).agg(total_sales=('amount', 'sum'), max_bill=('amount', 'max'), unique_customers=('customer_id','nunique'))
    
    # pass


# === ТЕСТЫ ===
sales_df = pd.DataFrame({
    "store_id": ["S1", "S1", "S2", "S1", "S2"],
    "amount": [100.0, 250.0, 500.0, 150.0, 300.0],
    "customer_id": [1, 2, 3, 1, 3]
})
summary = summarize_sales(sales_df)
assert list(summary.columns) == ["store_id", "total_sales", "max_bill", "unique_customers"], f"Неверные колонки: {list(summary.columns)}"
s1_row = summary[summary["store_id"] == "S1"].iloc[0]
assert s1_row["total_sales"] == 500.0 and s1_row["max_bill"] == 250.0 and s1_row["unique_customers"] == 2, f"Ошибка в расчетах для S1: {s1_row.to_dict()}"
print("✅ Задача 1 решена верно!")

✅ Задача 1 решена верно!


---
## Модуль 2. Оконная группировка: `.groupby().transform()`

### 🎯 Почему `transform` — любимый вопрос на собеседованиях:
* Обычный `groupby().agg()` **схлопывает** таблицу до числа уникальных групп.
* А `df.groupby('group')['val'].transform(func)` считает статистику по каждой группе и **растягивает (транслирует) её обратно на длину исходного `DataFrame`** (аналог оконной функции `AVG(val) OVER (PARTITION BY group)` в SQL или `keepdims=True` в NumPy)!
* Зачем это нужно в ML:
  1. **Доля от группы:** `df['val'] / df.groupby('group')['val'].transform('sum')`.
  2. **Центрирование внутри группы:** `df['val'] - df.groupby('group')['val'].transform('mean')`.
  3. **Умное заполнение пропусков средним/медианой по группе:**
     `df['val'] = df['val'].fillna(df.groupby('group')['val'].transform('median'))`

### 📚 Связанные методы в копилку:
* `df.groupby('group').filter(lambda g: len(g) >= 5)` — оставляет в исходной таблице только те строки, чья группа удовлетворяет условию (аналог `HAVING` в SQL).

### Задача 2
Напиши функцию `enrich_with_group_stats(df)`, которая принимает таблицу товаров с колонками `'category'` и `'price'` (в колонке `'price'` могут быть пропуски `NaN`!):
1. Создает копию `df = df.copy()`.
2. Заполняет пропуски `NaN` в колонке `'price'` **средним значением цены внутри соответствующей категории `'category'`** (используй `.fillna()` + `.transform('mean')`).
3. Добавляет колонку `'category_share'`, показывающую, какую долю составляет цена товара (после заполнения пропусков) от **суммарной стоимости всех товаров его категории** (`price / sum_of_prices_in_category`).
4. Возвращает обновленный `DataFrame`.

In [5]:
def enrich_with_group_stats(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df = df.copy()
    df['price'] = df['price'].fillna(df.groupby('category')['price'].transform('mean'))
    df['category_share'] = df['price'] / df.groupby('category')['price'].transform('sum')
    return df
    # pass


# === ТЕСТЫ ===
items_df = pd.DataFrame({
    "category": ["A", "A", "A", "B", "B"],
    "price": [10.0, 30.0, np.nan, 50.0, 50.0]
})
# В категории 'A' среднее = (10+30)/2 = 20.0. После заполнения цены в 'A': [10, 30, 20], сумма = 60.
enriched = enrich_with_group_stats(items_df)
assert not enriched["price"].isna().any(), "Пропуски в price не заполнены!"
assert np.isclose(enriched.loc[2, "price"], 20.0), f"Пропуск в группе A должен стать 20.0, получено: {enriched.loc[2, 'price']}"
assert np.allclose(enriched["category_share"], [10/60, 30/60, 20/60, 0.5, 0.5]), f"Неверные доли: {enriched['category_share'].tolist()}"
print("✅ Задача 2 решена верно!")

✅ Задача 2 решена верно!


---
## Модуль 3. Объединение таблиц по ключам (`pd.merge`)

### 🎯 Методы и параметры, которые нужны для задачи:
* `pd.merge(left_df, right_df, on='key', how='inner')` — объединяет две таблицы по общим колонкам (полный аналог `JOIN` в SQL).
* Основные режимы `how`:
  * `how='inner'` (по умолчанию) — оставляет только те ключи, которые есть **в обеих** таблицах.
  * `how='left'` — оставляет **все строки левой таблицы**, подтягивая к ним данные из правой (если совпадения в правой нет, подставятся `NaN`). Самый частый джойн в ML, чтобы не потерять объекты обучающей выборки!
  * `how='outer'` — объединяет вообще все ключи из обеих таблиц.
* Если колонки-ключи называются по-разному, используй `left_on='id_a', right_on='id_b'`.

### 📚 Связанные методы в копилку:
* Параметр `indicator=True` в `pd.merge` добавляет специальную колонку `'_merge'` со значениями `'both'`, `'left_only'`, `'right_only'` — удобно для поиска строк, которым не нашлось пары (`anti-join`).
* Параметр `validate='many_to_one'` (или `'one_to_one'`) проверяет уникальность ключей перед джойном и защищает от случайного размножения строк, если в справочнике оказались дубликаты!

### Задача 3
Даны две таблицы:
1. `orders` — таблица заказов с колонками `'order_id'`, `'user_id'`, `'amount'`.
2. `users` — справочник пользователей с колонками `'user_id'`, `'city'`.

Некоторых пользователей из `orders` может не оказаться в справочнике `users`.
Напиши функцию `join_orders_with_users(orders, users)`, которая:
1. Присоединяет к таблице `orders` информацию о городе пользователя через **`left` join** по `'user_id'` (чтобы ни один заказ не потерялся).
2. Заполняет пропуски в колонке `'city'` (для тех заказов, чей `user_id` не нашелся в справочнике) строкой `'Unknown_City'` и возвращает итоговую таблицу.

In [6]:
def join_orders_with_users(orders: pd.DataFrame, users: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    orders = pd.merge(orders,users,on='user_id',how='left')
    orders['city'] = orders["city"].fillna('Unknown_City')
    return orders
    # pass


# === ТЕСТЫ ===
ord_df = pd.DataFrame({
    "order_id": [10, 20, 30],
    "user_id": [1, 2, 99],
    "amount": [500, 1200, 300]
})
usr_df = pd.DataFrame({
    "user_id": [1, 2, 3],
    "city": ["Moscow", "SPb", "Sochi"]
})
merged = join_orders_with_users(ord_df, usr_df)
assert len(merged) == 3, "Количество заказов после left join не должно измениться!"
assert list(merged["city"]) == ["Moscow", "SPb", "Unknown_City"], f"Неверные города после merge: {list(merged['city'])}"
print("✅ Задача 3 решена верно!")

✅ Задача 3 решена верно!


---
## Модуль 4. Конкатенация таблиц (`pd.concat`)

### 🎯 Методы, которые нужны для задачи:
* `pd.concat([df1, df2, ...], axis=0, ignore_index=True)` — склеивает список таблиц:
  * При `axis=0` (по умолчанию) ставит таблицы **друг под другом** (вертикально, аналог `UNION ALL` в SQL, сопоставляя колонки по именам). Параметр **`ignore_index=True`** автоматически перенумерует индекс итоговой таблицы от `0` до `N-1`, чтобы не было дублирующихся индексов!
  * При `axis=1` ставит таблицы **рядом по горизонтали** (сопоставляя строки по их `index`).

### 📚 Важный нюанс при горизонтальной склейке (`axis=1`):
Если склеивать два `DataFrame` по `axis=1`, у которых **разные индексы** (например, после фильтрации одной из таблиц без `reset_index(drop=True)`), Pandas сопоставит их по меткам индекса и наставит `NaN` в несовпавших строках!

### Задача 4
Логи за январь (`jan_df`) и февраль (`feb_df`) хранятся в двух отдельных таблицах с одинаковыми колонками, но у каждой свой индекс (начинающийся с `0`). Кроме того, отдельно посчитана таблица новых признаков `features_df` для всех строк объединенного датасета (с индексом от `0` до `N_total - 1`).

Напиши функцию `combine_batches_and_features(jan_df, feb_df, features_df)`, которая:
1. Склеивает `jan_df` и `feb_df` **по вертикали** (`axis=0`) со сбросом индекса (`ignore_index=True`).
2. Приклеивает к полученной таблице справа `features_df` **по горизонтали** (`axis=1`) и возвращает итоговый `DataFrame`.

In [7]:
def combine_batches_and_features(jan_df: pd.DataFrame, feb_df: pd.DataFrame, features_df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df = pd.concat([jan_df,feb_df], axis=0, ignore_index=True)
    df = pd.concat([df,features_df],axis=1)
    return df
    # pass


# === ТЕСТЫ ===
jan = pd.DataFrame({"id": [1, 2], "val": [10, 20]})
feb = pd.DataFrame({"id": [3, 4], "val": [30, 40]})
feats = pd.DataFrame({"f1": [0.1, 0.2, 0.3, 0.4]})
full_df = combine_batches_and_features(jan, feb, feats)
assert full_df.shape == (4, 3), f"Ожидался размер (4, 3), получено: {full_df.shape}"
assert not full_df.isna().any().any(), "В таблице появились NaN! Забыл ignore_index=True при вертикальной склейке?"
assert list(full_df["f1"]) == [0.1, 0.2, 0.3, 0.4], "Неверные значения признака f1"
print("✅ Задача 4 решена верно!")

✅ Задача 4 решена верно!


---
## Модуль 5. Сводные таблицы (`pivot_table`)

### 🎯 Методы, которые нужны для задачи:
* `df.pivot_table(index='row_col', columns='col_col', values='val_col', aggfunc='sum', fill_value=0)` — превращает «длинную» таблицу логов в двумерную матрицу (сводную таблицу):
  * `index` — что пойдет в строки.
  * `columns` — уникальные значения какой колонки станут новыми столбцами.
  * `values` — какую числовую колонку агрегировать на пересечении.
  * `aggfunc` — функция агрегации (`'sum'`, `'mean'`, `'count'` и т.д., так как у одной пары `(index, column)` может быть несколько записей).
  * `fill_value=0` — чем сразу заполнить ячейки, на пересечении которых не было ни одной записи (вместо `NaN`).

### 📚 Связанные методы в копилку:
* `pd.crosstab(df['col1'], df['col2'], normalize=...)` — быстрая таблица сопряженности (считает частоты пересечения двух категориальных признаков, например, для матрицы ошибок confusion matrix).
* `df.pivot(...)` — работает без агрегации (падает с ошибкой, если в данных есть дубликаты пар `(index, columns)`), поэтому на практике `pivot_table` надежнее и универсальнее.

### Задача 5
В рекомендательных системах первый шаг — собрать матрицу интеракций `User-Item` из лога событий.

Напиши функцию `build_user_item_matrix(df)`, которая принимает лог покупок с колонками `'user_id'`, `'item_id'` и `'spent'` и с помощью `pivot_table` возвращает матрицу, где:
1. В строках (`index`) — `'user_id'`.
2. В столбцах (`columns`) — `'item_id'`.
3. В ячейках (`values`) — **сумма** (`'sum'`) потраченных денег `'spent'`.
4. Если пользователь не покупал товар, на пересечении должен стоять `0` (`fill_value=0`).

In [8]:
def build_user_item_matrix(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    return df.pivot_table(index='user_id',columns='item_id',values='spent',aggfunc='sum',fill_value=0)
    # pass


# === ТЕСТЫ ===
interactions = pd.DataFrame({
    "user_id": [1, 1, 2, 1],
    "item_id": ["A", "B", "A", "A"],
    "spent": [100, 200, 50, 150]
})
ui_mat = build_user_item_matrix(interactions)
assert ui_mat.shape == (2, 2), f"Неверная форма матрицы: {ui_mat.shape}"
assert ui_mat.loc[1, "A"] == 250, f"Сумма покупок товара A пользователем 1 должна быть 250 (100+150), получено: {ui_mat.loc[1, 'A']}"
assert ui_mat.loc[2, "B"] == 0, f"Пропущенное пересечение (user 2, item B) должно быть 0, получено: {ui_mat.loc[2, 'B']}"
print("✅ Задача 5 решена верно!")

✅ Задача 5 решена верно!


---
## Модуль 6. Распаковка списков (`explode`) и перевод из широкого формата в длинный (`melt`)

### 🎯 Методы, которые нужны для задачи:
* `df.explode('col', ignore_index=True)` — если в ячейках колонки `'col'` лежат **списки** (например, теги товара `['tech', 'sale']`), метод `explode` «разворачивает» каждый список так, что каждый элемент списка получает свою собственную строку (дублируя остальные поля строки)!
* `df.melt(id_vars=['id'], value_vars=['m1', 'm2'], var_name='month', value_name='sales')` — операция, обратная `pivot_table`: собирает несколько столбцов (`value_vars`) в две колонки — имя бывшего столбца (`var_name`) и его значение (`value_name`).

### 📚 Связанные методы в копилку:
* Часто перед `explode` строку с разделителями сначала превращают в список: `df['tags'] = df['tags'].str.split(',')`, а затем сразу делают `.explode('tags')`.

### Задача 6
В таблице фильмов `df` есть колонка `'movie'` и колонка `'genres'`, где жанры каждого фильма записаны в виде списка Python (например, `['Action', 'Sci-Fi']`).

Напиши функцию `count_movies_per_genre(df)`, которая:
1. Разворачивает списки в колонке `'genres'` на отдельные строки с помощью `.explode('genres')`.
2. Считает, сколько раз встретился каждый жанр (через `.value_counts()`), и возвращает полученный `Series`.

In [9]:
def count_movies_per_genre(df: pd.DataFrame) -> pd.Series:
    # ТВОЙ КОД ЗДЕСЬ
    df = df.explode('genres')
    return df['genres'].value_counts()
    # pass


# === ТЕСТЫ ===
movies = pd.DataFrame({
    "movie": ["Inception", "Matrix", "Interstellar"],
    "genres": [["Sci-Fi", "Action"], ["Sci-Fi", "Action"], ["Sci-Fi", "Drama"]]
})
genre_counts = count_movies_per_genre(movies)
assert genre_counts["Sci-Fi"] == 3 and genre_counts["Action"] == 2 and genre_counts["Drama"] == 1, f"Неверный подсчет жанров: {genre_counts.to_dict()}"
print("✅ Задача 6 решена верно!")

✅ Задача 6 решена верно!


---
## Модуль 7. Работа с датами и временем (`pd.to_datetime` и аксессор `.dt`)

### 🎯 Методы, которые нужны для задачи:
* `pd.to_datetime(df['date_str'])` — парсит строки или timestamp-числа в специальный тип `datetime64[ns]`.
* Аксессор **`.dt`** позволяет векторно извлекать любые календарные признаки из колонки типа `datetime`:
  * `df['dt'].dt.year`, `.dt.month`, `.dt.day`, `.dt.hour`.
  * `df['dt'].dt.dayofweek` (или `.dt.weekday`) — день недели от **`0` (Понедельник)** до **`6` (Воскресенье)**! Выходные — это `>= 5` (суббота `5` и воскресенье `6`).
* Разность двух дат `(df['end_dt'] - df['start_dt'])` возвращает тип `Timedelta`. Чтобы перевести её в целое число дней или секунд, используй `.dt.days` или `.dt.total_seconds()`.

### 📚 Связанные методы в копилку:
* `df.resample('D', on='dt')['sales'].sum()` — группировка временного ряда по дням (`'D'`), неделям (`'W'`), месяцам (`'ME'`) или часам (`'h'`).
* `pd.date_range(start='2026-01-01', end='2026-01-10', freq='D')` — генерация сетки дат.

### Задача 7
Напиши функцию `extract_date_features(df)`, которая принимает `DataFrame` со строковыми колонками `'order_date'` и `'delivery_date'` (формата `'YYYY-MM-DD'`):
1. Создает копию таблицы и переводит обе колонки в `datetime` через `pd.to_datetime`.
2. Создает колонку `'order_month'` — номер месяца заказа (`1..12`).
3. Создает булеву колонку `'is_weekend'` — `True`, если заказ сделан в субботу или воскресенье (`dayofweek >= 5`), иначе `False`.
4. Создает целочисленную колонку `'delivery_days'` — сколько дней заняла доставка (`(delivery_date - order_date).dt.days`).
5. Возвращает обновленный `DataFrame`.

In [10]:
def extract_date_features(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['delivery_date'] = pd.to_datetime(df['delivery_date'])

    df['order_month'] = df['order_date'].dt.month
    df['is_weekend'] = df['order_date'].dt.day_of_week >= 5
    df['delivery_days'] = (df['delivery_date'] - df['order_date']).dt.days
    return df
    # pass


# === ТЕСТЫ ===
# 2026-03-06 — пятница (4), 2026-03-07 — суббота (5)
dates_df = pd.DataFrame({
    "order_date": ["2026-03-06", "2026-03-07"],
    "delivery_date": ["2026-03-10", "2026-03-08"]
})
res_dt = extract_date_features(dates_df)
assert list(res_dt["order_month"]) == [3, 3], f"Неверный месяц: {list(res_dt['order_month'])}"
assert list(res_dt["is_weekend"]) == [False, True], f"Неверный флаг выходного дня: {list(res_dt['is_weekend'])}"
assert list(res_dt["delivery_days"]) == [4, 1], f"Неверное число дней доставки: {list(res_dt['delivery_days'])}"
print("✅ Задача 7 решена верно!")

✅ Задача 7 решена верно!


---
## Модуль 8. Временные ряды: сдвиги (`shift`), разности (`diff`) и групповые лаги

### 🎯 Методы, которые нужны для задачи:
* `df['col'].shift(1)` — сдвигает значения колонки на 1 шаг вниз (в текущей строке оказывается **предыдущее значение** — лаг `lag_1`, а в первой строке появляется `NaN`). При `shift(-1)` подтягивается следующее значение из будущего.
* `df['col'].diff(1)` — считает разность между текущим и предыдущим значением (`df['col'] - df['col'].shift(1)`).
* **Лаги внутри групп (`groupby` + `shift`/`diff`):** если в таблице лежат временные ряды сразу нескольких пользователей или магазинов, делать просто `df['sales'].shift(1)` **нельзя** (первая запись второго магазина подтянет последнюю запись первого!). Нужно обязательно группировать:
  * `df.groupby('store_id')['sales'].shift(1)`
  * `df.groupby('store_id')['sales'].diff(1)`

### 📚 Связанные методы в копилку:
* `df['col'].pct_change(1)` — относительное (процентное) изменение по сравнению с предыдущим шагом: `(x_t - x_{t-1}) / x_{t-1}` (постоянно используется в финансах для доходностей акций).

### Задача 8
Дана таблица ежедневных продаж магазинов `df`, уже отсортированная по `'store_id'` и дате, с колонками `'store_id'` и `'sales'`.

Напиши функцию `add_lag_and_diff(df)`, которая создает копию таблицы и добавляет две колонки **внутри каждого магазина `'store_id'` независимо**:
1. `'prev_day_sales'` — продажи этого же магазина в предыдущий день (`shift(1)` по группе `'store_id'`).
2. `'sales_diff'` — изменение продаж по сравнению с предыдущим днем этого же магазина (`diff(1)` по группе `'store_id'`).

In [13]:
def add_lag_and_diff(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df = df.copy()
    df['prev_day_sales'] = df.groupby('store_id')['sales'].shift(1)
    df['sales_diff'] = df.groupby('store_id')['sales'].diff(1)
    return df
    # pass


# === ТЕСТЫ ===
ts_df = pd.DataFrame({
    "store_id": ["A", "A", "A", "B", "B"],
    "sales": [100.0, 120.0, 90.0, 500.0, 550.0]
})
lagged = add_lag_and_diff(ts_df)
assert np.isnan(lagged.loc[0, "prev_day_sales"]) and np.isnan(lagged.loc[3, "prev_day_sales"]), "Первая запись каждого магазина должна иметь NaN в лаге!"
assert lagged.loc[1, "prev_day_sales"] == 100.0 and lagged.loc[4, "prev_day_sales"] == 500.0, "Неверные значения prev_day_sales"
assert lagged.loc[2, "sales_diff"] == -30.0 and lagged.loc[4, "sales_diff"] == 50.0, "Неверные значения sales_diff"
print("✅ Задача 8 решена верно!")

✅ Задача 8 решена верно!


---
## Модуль 9. Скользящие окна (`rolling`) и накопительные статистики (`cumsum`)

### 🎯 Методы, которые нужны для задачи:
* `df['col'].rolling(window=3, min_periods=1).mean()` — считает скользящее среднее по окну из последних `3` строк (текущая и две предыдущие).
  * По умолчанию первые `window - 1` значений равны `NaN` (пока окно не наберется полностью). Если указать **`min_periods=1`**, то для 1-й строки среднее посчитается по 1 элементу, для 2-й — по 2 элементам и т.д. (без `NaN` в начале!).
* `df['col'].cumsum()` — кумулятивная (накопительная) сумма от начала таблицы до текущей строки.
  * Для накопительной суммы внутри группы: `df.groupby('user_id')['spent'].cumsum()`.

### 📚 Связанные методы в копилку:
* `df['col'].expanding().mean()` — расширяющееся окно от самой первой строки до текущей (эквивалентно `rolling(window=len(df), min_periods=1)`).
* `df['col'].ewm(span=7).mean()` — экспоненциально взвешенное скользящее среднее (EMA), где свежие наблюдения имеют больший вес, чем старые.

### Задача 9
Напиши функцию `calc_rolling_and_cumulative(df)`, которая принимает `DataFrame` с колонкой ежедневной выручки `'revenue'`:
1. Создает копию таблицы.
2. Добавляет колонку `'rolling_3d_avg'` — скользящее среднее выручки с окном `window=3` и параметром `min_periods=1`.
3. Добавляет колонку `'cumulative_rev'` — накопительную сумму выручки от первого дня до текущего .
4. Возвращает результат.

In [18]:
def calc_rolling_and_cumulative(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df = df.copy()
    df['rolling_3d_avg'] = df['revenue'].rolling(3,1).mean()
    df['cumulative_rev'] = df['revenue'].cumsum()
    return df
    # pass


# === ТЕСТЫ ===
rev_df = pd.DataFrame({"revenue": [10.0, 20.0, 30.0, 40.0]})
res_win = calc_rolling_and_cumulative(rev_df)
assert np.allclose(res_win["rolling_3d_avg"], [10.0, 15.0, 20.0, 30.0]), f"Ошибка в скользящем среднем: {res_win['rolling_3d_avg'].tolist()}"
assert np.allclose(res_win["cumulative_rev"], [10.0, 30.0, 60.0, 100.0]), f"Ошибка в накопительной сумме: {res_win['cumulative_rev'].tolist()}"
print("✅ Задача 9 решена верно!")

✅ Задача 9 решена верно!


---
## Модуль 10. Босс-уровень с собеседований: Ранжирование внутри групп (`.rank`) и Top-N на группу

### 🎯 Классическая задача `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` в Pandas:
* Как в Pandas выбрать **Топ-2 самых дорогих товара в каждой категории** или пронумеровать строки внутри каждой группы по убыванию метрики?
* Способ 1 (через `.rank()`):
  `df['rank'] = df.groupby('category')['price'].rank(method='first', ascending=False)`
  * `ascending=False` дает ранг `1.0` самому большому значению.
  * `method='first'` гарантирует, что при одинаковой цене элементы получат уникальные ранги `1.0, 2.0, ...` в порядке их появления (полный аналог `ROW_NUMBER()` в SQL, тогда как `method='dense'` — это `DENSE_RANK()`).
* Способ 2 (через сортировку + `.groupby().head(k)` или `.cumcount()`):
  Если предварительно отсортировать таблицу `df.sort_values(['category', 'price'], ascending=[True, False])`, то:
  * `df.groupby('category').cumcount() + 1` — мгновенно проставит порядковые номера `1, 2, 3...` внутри каждой группы.
  * А `df.groupby('category').head(2)` — сразу оставит только первые 2 строки из каждой группы!

### Задача 10
Напиши функцию `get_top_2_per_department(df)`, которая принимает таблицу сотрудников с колонками `'department'`, `'name'` и `'salary'`:
1. Создает копию таблицы.
2. Добавляет колонку `'dept_rank'` (целое число `int` от `1` и выше) — ранг сотрудника по зарплате `'salary'` **по убыванию** внутри его отдела `'department'` (используй `.rank(method='first', ascending=False).astype(int)`).
3. Оставляет только сотрудников с рангом `<= 2` (Топ-2 в каждом отделе).
4. Сортирует результат по `'department'` (по возрастанию) и `'dept_rank'` (по возрастанию) и возвращает итоговый `DataFrame`.

In [19]:
def get_top_2_per_department(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df = df.copy()
    df['dept_rank'] = df.groupby('department')['salary'].rank(method='first', ascending=False).astype(int)
    df = df[df["dept_rank"] <= 2]
    df = df.sort_values(['department','dept_rank'])
    return df
    # pass


# === ТЕСТЫ ===
staff = pd.DataFrame({
    "department": ["IT", "IT", "IT", "Sales", "Sales", "Sales"],
    "name": ["Dev1", "Dev2", "Dev3", "Sal1", "Sal2", "Sal3"],
    "salary": [100, 300, 200, 150, 120, 180]
})
top_staff = get_top_2_per_department(staff)
assert len(top_staff) == 4, f"Должно остаться 4 сотрудника (по 2 из отдела), получено: {len(top_staff)}"
assert list(top_staff["name"]) == ["Dev2", "Dev3", "Sal3", "Sal1"], f"Неверный список Топ-2 сотрудников: {list(top_staff['name'])}"
assert list(top_staff["dept_rank"]) == [1, 2, 1, 2], f"Неверные ранги: {list(top_staff['dept_rank'])}"
print("🎉 ПОЗДРАВЛЯЮ! Весь курс тренажеров по NumPy и Pandas (40 из 40 задач) полностью пройден!")

🎉 ПОЗДРАВЛЯЮ! Весь курс тренажеров по NumPy и Pandas (40 из 40 задач) полностью пройден!
